# 第一周结束练习

为了证明您对 OpenAI API 和 Ollama 的熟悉程度，请构建一个可以回答技术问题的工具，  
并作出回应并作出解释。这是您在课程期间可以自己使用的工具！

In [ ]:
# 进口

from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import os

In [ ]:
# 常量

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [ ]:
# 设置环境

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

In [ ]:
# 这是问题；输入此内容以询问新问题

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
from openai import api_key

system_prompt = """you are a helpful assistant that can explain a technical question in a way that is easy to understand.
                    give a detailed explanation of the question and the logic behind it."""



In [ ]:
# 包装辅助变量和函数的工具辅助类

class TechnicalAssistant:
    def __init__(self, openai_api_key = None):
        self.openai_client = OpenAI(api_key=openai_api_key)
        self.ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
        self.model_gpt = MODEL_GPT
        self.model_llama = MODEL_LLAMA

    def _get_client_and_model(self, provider):
        # 如果 gpt 或在提供商中打开 ai
        if "gpt" in provider.lower():
            return self.openai_client, self.model_gpt
        elif "llama" in provider.lower():
            return self.openai_client, self.model_gpt
        else:
            raise ValueError(f"Unsupported provider: {provider}")

    def answer(self, question: str, provider: str="gpt"):
        client, model = self._get_client_and_model(provider)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# 让 gpt-4o-mini 接听，带流媒体

assistant = TechnicalAssistant()

assistant.answer(question, "gpt")



In [ ]:
# 让 Llama 3.2 来回答

assistant.answer(question, "ollama")